# KHUDA 10기 ML세션
Date : 2026.08.12 (수)

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [6]:
### 건들지마세요
n=3
answer = [0] * 4

## Task 0

초기 설정을 진행합니다.

별도의 정답은 없지만, 완료되지 않으면 이후 Task 수행이 불가능합니다.

* sklearn.datasets.load_iris()로 데이터를 불러오세요.
* feature(X) : iris.data (4개 컬럼: 꽃받침 길이/너비, 꽃잎 길이/너비)
* target(y) : iris.target # 참고사항 : (0=setosa, 1=versicolor, 2=virginica) 입니다
* feature(X)에 대해 StandardScaler()를 적용해주세요.


In [7]:

from sklearn.datasets import load_iris

iris = load_iris()

X = iris.data
y = iris.target

iris_df = pd.DataFrame(data=X, columns= iris.feature_names)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)





df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target
print(df.head())


   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   target  
0       0  
1       0  
2       0  
3       0  
4       0  


## Task 1

sigmoid, relu 함수를 numpy만 사용해서 직접 구현하세요. (softmax 함수는 제공합니다.)

a = np.array([1.0, 2.0, 3.0])에 sigmoid, relu, softmax를 적용한 뒤, 세 가지 함수를 모두 적용한 값들을 합하여 소수 셋째 자리에서 반올림하여 둘째 자리까지 구해 answer[2]에 저장합니다.

In [8]:
## Input Box

ans = 0
a = np.array([1.0, 2.0, 3.0])



# 이 함수는 건들지 마세요.
def softmax(x):
    if x.ndim == 1:
        x = x - np.max(x)
        exp_x = np.exp(x)
        return exp_x / np.sum(exp_x)
    else:
        x = x - np.max(x, axis=-1, keepdims=True)
        exp_x = np.exp(x)
        return exp_x / np.sum(exp_x, axis=-1, keepdims=True)


def sigmoid(x) :
  h = (1+np.exp(-x))**(-1)
  return h


def relu(x) :
  return np.maximum(0,x)


i = softmax(a)
j = sigmoid(a)
k = relu(a)

b = (i+k+j).sum()

ans = np.round(b,2)


In [9]:
answer[1] = ans
print(answer[1])

9.56


## Task 2

2층 신경망을 TwoLayerNet 클래스 하나로 구현하세요. 교재 137페이지 [4.5.1 2층 신경망 클래스 구현하기] 를 참고해주세요.


```
__init__(input_size, hidden_size, output_size, weight_init_std=1.0, seed=42)

```

```
predict(x)
→ a1 = x@W1+b1 → z1 = sigmoid(a1) → a2 = z1@W2+b2 → y = softmax(a2) 반환 순으로 진행합니다.

```

TwoLayerNet(4, 8, 3, weight_init_std=1.0, seed=42)를 만들어 X_scaled를 predict한 뒤, 0번째 샘플 확률 중 최댓값을 반올림(둘째 자리)해서 answer[2]에 저장하세요.

In [11]:
## Input Box
ans = 0

def to_one_hot(y, num_classes):
    T = np.zeros((y.size, num_classes))
    for idx, row in enumerate(T):
        row[y[idx]] = 1
    return T

def cross_entropy_error(y, t):
    if y.ndim == 1:
        t = t.reshape(1, t.size)
        y = y.reshape(1, y.size)
    delta = 1e-7
    batch_size = y.shape[0]
    return -np.sum(t * np.log(y + delta)) / batch_size

## 아래에 클래스를 구현해주시면 됩니다.

class TwoLayerNet:
    def __init__(self, input_size, hidden_size, output_size, weight_init_std=1.0, seed=42):
        rng = np.random.RandomState(seed)

        self.params ={}
        self.params['W1'] = weight_init_std *\
                              rng.randn(input_size, hidden_size)
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = weight_init_std *\
                              rng.randn(hidden_size, output_size)
        self.params['b2'] = np.zeros(output_size)


    def predict(self, x):
      W1, W2 = self.params['W1'] , self.params['W2']
      b1, b2 = self.params['b1'] , self.params['b2']

      a1 = np.dot(x,W1) + b1
      z1 = sigmoid(a1)
      a2 = np.dot(z1, W2) + b2
      y = softmax(a2)

      return y

    def loss(self, x, t):
      y = self.predict(x)

      return cross_entropy_error(y,t)


    def accuracy(self, x, t):
      y = self.predict(x)
      y = np.argmax(y , axis=1)
      t = np.argmax(t , axis=1)

      acc = np.sum(y==t)/ float(x.shape[0])
      return acc




## 이 부분은 건들지 마세요.
net = TwoLayerNet(4, 8, 3, weight_init_std=1.0, seed=42)
t_onehot = to_one_hot(y, 3)
y_pred = net.predict(X_scaled)
ans = round(float(np.max(y_pred[0])), 2)

In [12]:
answer[2] = ans
print(answer[2])

0.48


## Task 3

numerical_gradient를 구현하고, 전체 데이터로 1번만 파라미터를 업데이트한 뒤 loss가 줄어드는지 확인합니다.

numerical_gradient(f, x) : 중심차분 방식 (h=1e-4)



---


TwoLayerNet.numerical_gradient(self, x, t) : W1, b1, W2, b2 기울기를 grads 딕셔너리로 반환


net.numerical_gradient(X_scaled, t_onehot)으로 기울기를 구하고, learning_rate=0.1로 W1, b1, W2, b2를 한 번만 업데이트하세요.
업데이트 후 net.loss(X_scaled, t_onehot)을 반올림(둘째 자리)해서 answer[5]에 저장하세요.



---



Hint : 아래와 단계로 진행합니다.
1. net_numerical_gradient 함수를 구현해주세요 (교재 p.138을 참고해 grads를 반환합니다.)
2. TwoLayerNet.numerical_gradient = net_numerical_gradient 를 통해 기존 클래스에 정의한 메서드로를 추가합니다.
3. grad = net.numerical_gradient(X_scaled, t_onehot) 를 통해 W1, b1, W2, b2에 대한 기울기를 한 번에 계산합니다.
4. learning_rate=0.1로 W1, b1, W2, b2 각각을 한 번만 업데이트합니다.
5. 업데이트된 net으로 net.loss(X_scaled, t_onehot)을 다시 계산하고, 소수 셋째 자리에서 반올림하여 answer[5]에 저장합니다.

In [17]:
## Input Box

ans = 0
learning_rate = 0.1
net = TwoLayerNet(4, 8, 3, weight_init_std=1.0, seed=42)

def numerical_gradient(f, x):
    h = 1e-4
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        tmp_val = x[idx]
        x[idx] = tmp_val + h
        fxh1 = f(x)
        x[idx] = tmp_val - h
        fxh2 = f(x)
        grad[idx] = (fxh1 - fxh2) / (2 * h)
        x[idx] = tmp_val
        it.iternext()
    return grad

## 이 밑부분부터 구현해주세요
def net_numerical_gradient(self, x, t):
  loss_W = lambda W: self.loss(x, t)

  grads = {}
  grads['W1'] = numerical_gradient(loss_W,self.params['W1'])
  grads['b1'] = numerical_gradient(loss_W,self.params['b1'])
  grads['W2'] = numerical_gradient(loss_W,self.params['W2'])
  grads['b2'] = numerical_gradient(loss_W,self.params['b2'])

  return grads

TwoLayerNet.numerical_gradient = net_numerical_gradient
grad = net.numerical_gradient(X_scaled, t_onehot)

for i in ('W1', 'b1', 'W2', 'b2'):
    net.params[i] -= learning_rate * grad[i]

loss_val = net.loss(X_scaled, t_onehot)
ans = round(float(loss_val), 2)

In [18]:
answer[3] = ans
print(answer[3])

1.57


## Task 4 (추가문제 : 별도의 정답은 없습니다. 앞의 3문제를 모두 완료했다면 탈출이 가능합니다)
다음은,
데이터를 학습용(train) : 평가용(test) = 8 : 2로 나누고, TwoLayerNet을 학습 데이터로만 미니배치 학습시킨 뒤 평가 데이터로 최종 성능을 확인하는 과정입니다.

다음 빈 부분을 채워주세요! 그래프가 나오면 성공입니다.


데이터 분리: train_test_split을 진행합니다.
`(X_scaled, y, test_size=0.2, random_state=42, stratify=y)`



원-핫 인코딩: `to_one_hot(Task2의 함수)으로 t_train, t_test`를 만드세요.
to_one_hot(y, num_classes)의 num_classes는 "원-핫 벡터의 길이 = 클래스가 총 몇 개인지" 를 나타냅니다. 이 데이터셋에선 붓꽃의 종류가 0,1,2로 3개가 있습니다.




In [ ]:
## Input Box

## 여기에 구현해주시면 됩니다


## 이 밑 부분은 건들지 마세요
net = TwoLayerNet(4, 8, 3, weight_init_std=1.0, seed=42)

np.random.seed(42)
iters_num = 200
train_size = X_train.shape[0]
batch_size = 20
learning_rate = 0.1
iter_per_epoch = max(train_size / batch_size, 1)

train_loss_list = []
train_acc_list = []
test_acc_list = []

for i in range(iters_num):
    batch_mask = np.random.choice(train_size, batch_size)
    x_batch = X_train[batch_mask]
    t_batch = t_train[batch_mask]

    grad = net.numerical_gradient(x_batch, t_batch)
    for key in ('W1', 'b1', 'W2', 'b2'):
        net.params[key] -= learning_rate * grad[key]

    train_loss_list.append(net.loss(x_batch, t_batch))

    if i % iter_per_epoch == 0:
        train_acc_list.append(net.accuracy(X_train, t_train))
        test_acc_list.append(net.accuracy(X_test, t_test))

# ---- 그래프 ----
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(train_loss_list)
axes[0].set_xlabel("iteration"); axes[0].set_ylabel("loss"); axes[0].set_title("Training loss")

axes[1].plot(train_acc_list, marker='o', label='train')
axes[1].plot(test_acc_list, marker='s', label='test')
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].set_ylim(0, 1.0)
axes[1].set_title("Train vs Test accuracy"); axes[1].legend()
plt.tight_layout()
plt.show()

# 정답 확인!

In [19]:
print("=== 작성하신 정답 ===")
for i in range (1, (n+1)) : print("Task " + str(i) + " :: " + str(answer[i]))

=== 작성하신 정답 ===
Task 1 :: 9.56
Task 2 :: 0.48
Task 3 :: 1.57


In [20]:
### 정답 채점 코드
import hashlib

ANSWER_URL = "https://github.com/chaegyeong/KHUDA_ML_10th/raw/4cf2f86ca09373cd0b58387d2795bc85824ecf5b/KHUDA_ML10th_WEEK5/answer.csv"

def md5 (s) : return hashlib.md5(s.encode("utf-8")).hexdigest()

ans_df = pd.read_csv(ANSWER_URL)
gt = {int(r["task"]): str(r["answer"]).strip().lower()
      for _, r in ans_df.iterrows()}

wrong = []

for i in range(1, (n+1)) :
    user_answer = "" if answer[i] is None else md5(str(answer[i]).strip().lower())
    if (user_answer != gt.get(i, "")) : wrong.append(i)

print("탈출!" if not wrong else f"틀린번호 : {wrong}")

탈출!
